# ID3 decision tree, written from scratch

The Car Evaluation dataset: 1,728 cars described by six categorical attributes, each rated
unacceptable / acceptable / good / very good.

No machine learning library is used. Entropy, information gain, the recursive tree induction and
the classifier are all written here. `03_vs_sklearn.ipynb` then puts this tree next to
scikit-learn's and works out why the two disagree.

## Step 1: Load the data

`car.data` has no header row, so the column names come from the accompanying `car.names` file:
`buying, maint, doors, persons, lug_boot, safety, class`.

In [1]:
import pandas as pd

columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv('car.data', header=None, names=columns)

print(df.shape)
df.head()

(1728, 7)


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


## Step 2: Are any attributes continuous?

ID3 splits on discrete values, so any continuous attribute would have to be binned first. Checking
the dtype and the unique values of each column settles it.

In [2]:
print(df.dtypes)
print()
for col in df.columns:
    print(col, ':', sorted(df[col].unique()))

buying      object
maint       object
doors       object
persons     object
lug_boot    object
safety      object
class       object
dtype: object

buying : ['high', 'low', 'med', 'vhigh']
maint : ['high', 'low', 'med', 'vhigh']
doors : ['2', '3', '4', '5more']
persons : ['2', '4', 'more']
lug_boot : ['big', 'med', 'small']
safety : ['high', 'low', 'med']
class : ['acc', 'good', 'unacc', 'vgood']


All seven columns are `object`, and each attribute has only three or four distinct values.

`doors` is the one worth a second look: it reads as numeric until you see `'5more'` in it. That
makes it an ordinal category, not a number - and it is why the column arrives as `object` rather
than `int`. A quick glance at the dtype would have suggested "needs converting"; the values say
otherwise.

**No attribute is continuous, so no binning is needed.**

## Step 3: Splitting into train and test

Without `train_test_split`, the split is three lines:

1. `df.sample(frac=1, random_state=42)` shuffles every row.
2. Cut at 80%.
3. First part trains, the rest tests.

`random_state` is fixed so the split is the same on every run - otherwise the accuracy at the end
would move around and could not be compared with anything.

The class distribution is printed for both halves as a check. This is not stratified sampling, so
the ratios only match because the shuffle happened to be even; on a smaller or more skewed dataset
that would need handling.

In [3]:
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_point = int(len(shuffled_df) * 0.8)
train_df = shuffled_df.iloc[:split_point].reset_index(drop=True)
test_df = shuffled_df.iloc[split_point:].reset_index(drop=True)

print('Train size:', train_df.shape)
print('Test size:', test_df.shape)
print()
print('Train class distribution:')
print(train_df['class'].value_counts(normalize=True))
print()
print('Test class distribution:')
print(test_df['class'].value_counts(normalize=True))

Train size: (1382, 7)
Test size: (346, 7)

Train class distribution:
class
unacc    0.711288
acc      0.217077
vgood    0.036179
good     0.035456
Name: proportion, dtype: float64

Test class distribution:
class
unacc    0.656069
acc      0.242775
good     0.057803
vgood    0.043353
Name: proportion, dtype: float64


## Step 4: Building the tree

Four pieces, in order:

1. `entropy(labels)` - how mixed a set of class labels is: `E = -sum(p_i * log2(p_i))`. Zero when
   every row has the same label, highest when they are evenly spread.
2. `information_gain(data, attribute, target_col)` - `IG(S,A) = E(S) - sum_v (|Sv|/|S|) * E(Sv)`.
   How much entropy drops if the data is split on that attribute.
3. `best_attribute(...)` - the attribute with the largest gain.
4. `build_tree(...)` - recursion. Split on the best attribute, remove it from the list, recurse on
   each branch. Stop when the node is pure, or when no attributes are left (then the majority
   class).

One detail that is easy to get wrong: `attribute_values` is computed once from the **whole**
dataset and passed down. Without it, a branch value that happens not to appear in the current
subset would get no subtree at all, and a test row carrying that value would hit a dead end. With
it, every possible value gets a branch - an empty one falls back to the parent's majority class.

Note also what the recursion implies. **Splitting on an attribute uses it up.** With six
attributes, the tree cannot go deeper than six levels. That constraint comes back in
`03_vs_sklearn.ipynb`.

**Source acknowledgment:** The overall design of this ID3 implementation (Shannon entropy, information gain, a recursive tree-induction function that returns a nested dictionary, and a separate classification function that walks the dictionary) is adapted from the course lab example `Lab4_decision_tree.ipynb`, which itself is based on:

> Peter Harrington, *Machine Learning in Action*, Chapter 3 (Decision Trees).

The code here was rewritten by me to operate on pandas DataFrames (instead of Python lists) and adapted for the Car Evaluation dataset.


In [4]:
import math

def entropy(labels):
    """Shannon entropy of a list/Series of class labels."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = labels.value_counts()
    ent = 0.0
    for count in counts:
        p = count / n
        ent -= p * math.log2(p)
    return ent


def information_gain(data, attribute, target_col):
    """Information gain from splitting `data` on `attribute`."""
    base_entropy = entropy(data[target_col])
    n = len(data)
    weighted_entropy = 0.0
    for value, subset in data.groupby(attribute):
        weighted_entropy += (len(subset) / n) * entropy(subset[target_col])
    return base_entropy - weighted_entropy


def best_attribute(data, attributes, target_col):
    """Return the attribute (from `attributes`) with the highest information gain."""
    gains = {attr: information_gain(data, attr, target_col) for attr in attributes}
    return max(gains, key=gains.get)


def majority_class(data, target_col):
    """Return the most frequent class label."""
    return data[target_col].value_counts().idxmax()

def build_tree(data, attributes, target_col, attribute_values):
    """
    Tree induction function (ID3).
    - data: current subset of the training DataFrame at this node
    - attributes: list of attribute names still available to split on
    - target_col: name of the class column
    - attribute_values: dict {attribute: [all possible values]} computed once from
      the full training set, so every branch value gets a subtree even if some
      value is missing in the current subset.
    Returns a leaf (class label string) or a nested dict {attribute: {value: subtree}}.
    """
    # Stopping condition 1: all rows have the same class
    if data[target_col].nunique() == 1:
        return data[target_col].iloc[0]

    # Stopping condition 2: no attributes left to split on
    if len(attributes) == 0:
        return majority_class(data, target_col)

    # Choose the best attribute to split on
    attr = best_attribute(data, attributes, target_col)
    remaining_attributes = [a for a in attributes if a != attr]

    tree = {attr: {}}
    parent_majority = majority_class(data, target_col)

    for value in attribute_values[attr]:
        subset = data[data[attr] == value]
        if len(subset) == 0:
            # No training rows with this value at this node -> fall back to majority class
            tree[attr][value] = parent_majority
        else:
            tree[attr][value] = build_tree(subset, remaining_attributes, target_col, attribute_values)

    return tree

In [5]:
target_col = 'class'
attributes = [c for c in train_df.columns if c != target_col]
attribute_values = {attr: sorted(df[attr].unique()) for attr in attributes}

tree = build_tree(train_df, attributes, target_col, attribute_values)
tree

{'safety': {'high': {'persons': {'2': 'unacc',
    '4': {'buying': {'high': {'maint': {'high': 'acc',
        'low': 'acc',
        'med': 'acc',
        'vhigh': 'unacc'}},
      'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
          'med': {'doors': {'2': 'acc',
            '3': 'acc',
            '4': 'vgood',
            '5more': 'acc'}},
          'small': 'acc'}},
        'low': {'doors': {'2': 'good',
          '3': 'vgood',
          '4': 'vgood',
          '5more': {'lug_boot': {'big': 'vgood',
            'med': 'vgood',
            'small': 'good'}}}},
        'med': {'lug_boot': {'big': 'vgood',
          'med': {'doors': {'2': 'good',
            '3': 'good',
            '4': 'vgood',
            '5more': 'vgood'}},
          'small': 'good'}},
        'vhigh': 'acc'}},
      'med': {'maint': {'high': 'acc',
        'low': {'lug_boot': {'big': 'vgood',
          'med': {'doors': {'2': 'good',
            '3': 'good',
            '4': 'good',
            '5more':

## Step 5: Classifying and scoring

`classify()` walks one test row from the root down. At each node it reads that node's attribute
from the row and follows the matching branch, until it lands on a leaf holding a class label.

Because the tree was built with `attribute_values` covering every category, every branch already
exists and the walk cannot fall off the tree. The `default_class` fallback covers the remaining
case - a value never seen anywhere in the dataset - by returning the training majority class
instead of raising.

In [6]:
train_majority_class = majority_class(train_df, target_col)

def classify(tree, sample, default_class=train_majority_class):
    """
    Classification function.
    - tree: the nested dict returned by build_tree() (or a leaf/class label)
    - sample: a pandas Series (one row) with attribute values to classify
    Returns the predicted class label.
    """
    if not isinstance(tree, dict):
        return tree  # reached a leaf

    attr = next(iter(tree))
    value = sample[attr]

    if value not in tree[attr]:
        return default_class  # safety net for unseen values

    return classify(tree[attr][value], sample, default_class)

In [7]:
predictions = test_df.apply(lambda row: classify(tree, row), axis=1)
actual = test_df[target_col]

accuracy = (predictions == actual).mean()
print('Test accuracy:', accuracy)

pd.DataFrame({'actual': actual, 'predicted': predictions}).head(10)

Test accuracy: 0.8757225433526011


,actual,predicted
0,acc,acc
1,unacc,unacc
2,acc,acc
3,acc,acc
4,unacc,unacc
5,unacc,unacc
6,unacc,unacc
7,good,good
8,unacc,unacc
9,unacc,unacc
